# Notebook 2 — Xây dựng Ma trận (.npz)

**Input**: artifacts từ `subsample/` (được tạo bởi Notebook 1)

**Output**: `user_outfit_adj.npz`, `outfit_item_adj.npz`, `category_cooccurrence_matrix.npz`, `item_item_matrix.npz`

> Notebook này đọc trực tiếp từ `item_sub.csv`, `outfit_sub.csv`, `user_sub.csv`, `train_uo_sub.csv`
> (đã được lọc active-user ở Notebook 1) — **không load lại raw data gốc**.

## 0. Cài đặt

In [1]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('✅ Dependencies OK')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.3 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.1 MB/s eta 0:00:00
✅ Tất cả thư viện đã được cài đặt!


## 1. Imports & Setup

In [2]:
import numpy as np
import pandas as pd
import os, random, json
import networkx as nx
from collections import defaultdict
from scipy.sparse import dok_matrix, save_npz, load_npz

SEED = 42
random.seed(SEED); np.random.seed(SEED)

def parse_ids(s):
    s = str(s).strip()
    sep = ';' if ';' in s else ' '
    return [int(x.strip()) for x in s.split(sep) if x.strip()]

print('✅ Imports OK')


Device: cuda
GPU: Tesla T4


In [3]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
KAGGLE_DATA = Path('/kaggle/input/datasets/kiettruonglifeez/recsys-fgat')
LOCAL_DATA  = REPO_ROOT / 'hfgat_rewrite_validate' / 'Dataset'

if KAGGLE_DATA.exists():
    DATA_ROOT   = KAGGLE_DATA
    OUTPUT_ROOT = Path('/kaggle/working')
    IMAGE_ROOT  = OUTPUT_ROOT / 'images'
elif LOCAL_DATA.exists():
    DATA_ROOT   = LOCAL_DATA
    OUTPUT_ROOT = REPO_ROOT / 'output_fgat_active_user'
    IMAGE_ROOT  = LOCAL_DATA / 'fashion_item_images'
else:
    raise FileNotFoundError(
        f'Dataset not found. Expected {LOCAL_DATA} (local) or {KAGGLE_DATA} (Kaggle).'
    )

DATA_PATH   = str(DATA_ROOT) + '/'
OUTPUT_PATH = str(OUTPUT_ROOT) + '/'
IMAGE_DIR   = str(IMAGE_ROOT) + '/'

for _sub in ('embeddings', 'matrices', 'subsample', 'splits', 'models', 'images'):
    (OUTPUT_ROOT / _sub).mkdir(parents=True, exist_ok=True)

def file_exists(path):
    exists = os.path.exists(path)
    status = '✅ Đã có' if exists else '⏳ Chưa có'
    print(f'  {status}: {os.path.basename(path)}')
    return exists

print('✅ Paths OK')
print(f'  OUTPUT_PATH = {OUTPUT_PATH}')


✅ Paths OK


## Stage 1 — Load subsample artifacts từ Notebook 1

Đọc từ `subsample/` thay vì raw data gốc — đảm bảo nhất quán với filter đã làm ở Notebook 1.

In [ ]:
print('=== STAGE 1: Load subsample artifacts ===')

from pathlib import Path
REPO_ROOT = Path.cwd().resolve()

NB1_SUB_CANDIDATES = [
    str(REPO_ROOT / 'output_fgat_active_user' / 'subsample') + '/',
    '/kaggle/input/notebooks/kiettruonglifeez/fgat-session1-active-user/subsample/',
    '/kaggle/input/fgat-session1-active-user/subsample/',
    '/kaggle/working/subsample/',
]

SUB_DIR = None
for p in NB1_SUB_CANDIDATES:
    if os.path.exists(p + 'item_sub.csv'):
        SUB_DIR = p
        break

if SUB_DIR is None:
    search_roots = [str(REPO_ROOT), '/kaggle/input']
    print('❌ Không tìm thấy subsample. CSV files found:')
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for r, _, files in os.walk(root):
            for fn in files:
                if fn.endswith('.csv'):
                    print(f'  {os.path.join(r, fn)}')
    raise FileNotFoundError('Chạy Notebook 1 trước!')

print(f'✅ Tìm thấy subsample tại: {SUB_DIR}')

required_files = ['item_sub.csv', 'outfit_sub.csv', 'user_sub.csv', 'train_uo_sub.csv']
for fn in required_files:
    assert os.path.exists(SUB_DIR + fn), f'❌ Không tìm thấy {fn}'

item_data   = pd.read_csv(SUB_DIR + 'item_sub.csv')
outfit_data = pd.read_csv(SUB_DIR + 'outfit_sub.csv')
user_data   = pd.read_csv(SUB_DIR + 'user_sub.csv')
train_uo    = pd.read_csv(SUB_DIR + 'train_uo_sub.csv')

item_data['item_id']     = item_data['item_id'].astype(int)
outfit_data['outfit_id'] = outfit_data['outfit_id'].astype(int)
user_data['user_id']     = user_data['user_id'].astype(int)
train_uo['user_id']      = train_uo['user_id'].astype(int)
train_uo['outfit_id']    = train_uo['outfit_id'].astype(int)

stats_path = SUB_DIR + 'filter_stats.json'
if os.path.exists(stats_path):
    with open(stats_path) as f:
        stats = json.load(f)
    print(f'  Filter stats: {stats}')

print(f'  Items   : {len(item_data):,} | Categories: {item_data["category"].nunique()}')
print(f'  Outfits : {len(outfit_data):,}')
print(f'  Users   : {len(user_data):,}')
print(f'  Edges   : {len(train_uo):,}')
print(item_data.head(3))
print(outfit_data.head(3))
print('✅ Stage 1 hoàn thành!')


In [6]:
print('=== STAGE 3A: Xây dựng ma trận adjacency ===')

USER_OUTFIT_FILE = OUTPUT_PATH + 'matrices/user_outfit_adj.npz'
OUTFIT_ITEM_FILE = OUTPUT_PATH + 'matrices/outfit_item_adj.npz'

if file_exists(USER_OUTFIT_FILE) and file_exists(OUTFIT_ITEM_FILE):
    user_outfit_adj = load_npz(USER_OUTFIT_FILE)
    outfit_item_adj = load_npz(OUTFIT_ITEM_FILE)
    print(f'  user_outfit_adj: {user_outfit_adj.shape}')
    print(f'  outfit_item_adj: {outfit_item_adj.shape}')
else:
    item_ids_uniq   = sorted(item_data['item_id'].unique())
    outfit_ids_uniq = sorted(outfit_data['outfit_id'].unique())
    user_ids_uniq   = sorted(user_data['user_id'].unique())

    user_index   = {uid: i for i, uid in enumerate(user_ids_uniq)}
    outfit_index = {oid: i for i, oid in enumerate(outfit_ids_uniq)}
    item_index   = {iid: i for i, iid in enumerate(item_ids_uniq)}

    # --- User-Outfit adjacency (từ train_uo_sub) ---
    print('  Building user_outfit_adj từ train_uo_sub...')
    user_outfit_adj = dok_matrix((len(user_ids_uniq), len(outfit_ids_uniq)), dtype=np.int8)
    for _, row in train_uo.iterrows():
        u_id = int(row['user_id'])
        o_id = int(row['outfit_id'])
        if u_id in user_index and o_id in outfit_index:
            user_outfit_adj[user_index[u_id], outfit_index[o_id]] = 1
    user_outfit_adj = user_outfit_adj.tocsr()
    save_npz(USER_OUTFIT_FILE, user_outfit_adj)
    print(f'  💾 Đã lưu user_outfit_adj: {user_outfit_adj.shape}, nnz={user_outfit_adj.nnz:,}')

    # --- Outfit-Item adjacency ---
    print('  Building outfit_item_adj...')
    outfit_item_adj = dok_matrix((len(outfit_ids_uniq), len(item_ids_uniq)), dtype=np.int8)
    for _, row in outfit_data.iterrows():
        o_id = int(row['outfit_id'])
        for i_id in parse_ids(row['items']):
            if i_id in item_index:
                outfit_item_adj[outfit_index[o_id], item_index[i_id]] = 1
    outfit_item_adj = outfit_item_adj.tocsr()
    save_npz(OUTFIT_ITEM_FILE, outfit_item_adj)
    print(f'  💾 Đã lưu outfit_item_adj: {outfit_item_adj.shape}, nnz={outfit_item_adj.nnz:,}')

print('✅ Stage 3A hoàn thành!')
print('\n📋 Adjacency matrices check:')
print(f'  user_outfit_adj : shape={user_outfit_adj.shape}, nnz={user_outfit_adj.nnz:,}')
print(f'  outfit_item_adj : shape={outfit_item_adj.shape}, nnz={outfit_item_adj.nnz:,}')
print(f'  Density user_outfit: {user_outfit_adj.nnz / (user_outfit_adj.shape[0]*user_outfit_adj.shape[1]):.6f}')
print(f'  Density outfit_item: {outfit_item_adj.nnz / (outfit_item_adj.shape[0]*outfit_item_adj.shape[1]):.6f}')

=== STAGE 3A: Xây dựng ma trận adjacency ===
  ⏳ Chưa có: user_outfit_adj.npz
  Building user_outfit_adj từ train_uo_sub...
  💾 Đã lưu user_outfit_adj: (25263, 6622), nnz=127,536
  Building outfit_item_adj...
  💾 Đã lưu outfit_item_adj: (6622, 14419), nnz=26,047
✅ Stage 3A hoàn thành!

📋 Adjacency matrices check:
  user_outfit_adj : shape=(25263, 6622), nnz=127,536
  outfit_item_adj : shape=(6622, 14419), nnz=26,047
  Density user_outfit: 0.000762
  Density outfit_item: 0.000273


## Stage 3B — Category Co-occurrence

In [7]:
print('=== STAGE 3B: Tính Category Co-occurrence Weights ===')

CATEGORY_MTX_FILE = OUTPUT_PATH + 'matrices/category_cooccurrence_matrix.npz'
CATEGORY_W_FILE   = OUTPUT_PATH + 'matrices/normalized_category_weights.npy'

if file_exists(CATEGORY_W_FILE):
    normalized_category_weights = np.load(CATEGORY_W_FILE, allow_pickle=True).item()
    print(f'  Đã load {len(normalized_category_weights):,} category weight pairs')
else:
    item_data_str = item_data.copy()
    item_data_str['item_id'] = item_data_str['item_id'].astype(str)
    item_category_map = dict(zip(item_data_str['item_id'], item_data_str['category']))

    category_cooccurrence = defaultdict(int)
    category_count        = defaultdict(int)

    print('  Đang tính co-occurrence trên outfit_data_filtered...')
    for _, row in outfit_data.iterrows():
        items_list = [str(x) for x in parse_ids(row['items'])]
        categories = list(set(
            item_category_map[item] for item in items_list if item in item_category_map
        ))
        for c in categories:
            category_count[c] += 1
        for i in range(len(categories)):
            for j in range(i + 1, len(categories)):
                category_cooccurrence[(categories[i], categories[j])] += 1
                category_cooccurrence[(categories[j], categories[i])] += 1

    print('  Đang tính edge weights...')
    category_weights = {}
    cat_keys = list(category_count.keys())
    for (c1, c2), g_cc in category_cooccurrence.items():
        g_c   = category_count[c1]
        denom = sum(
            category_cooccurrence.get((c1, c), 0) / max(category_count.get(c, 1), 1)
            for c in cat_keys
        )
        category_weights[(c1, c2)] = (g_cc / g_c) / denom if denom != 0 else 0

    all_w   = list(category_weights.values())
    min_w, max_w = min(all_w), max(all_w)
    normalized_category_weights = {
        k: (v - min_w) / (max_w - min_w) if max_w != min_w else 0.0
        for k, v in category_weights.items()
    }

    np.save(CATEGORY_W_FILE, normalized_category_weights)
    print(f'  💾 Đã lưu normalized_category_weights: {len(normalized_category_weights):,} pairs')

    unique_cats = sorted(set(c for pair in normalized_category_weights for c in pair))
    n_cats      = len(unique_cats)
    cat_to_idx  = {c: i for i, c in enumerate(unique_cats)}
    cooc_matrix = dok_matrix((n_cats, n_cats), dtype=np.float64)
    for (c1, c2), w in normalized_category_weights.items():
        cooc_matrix[cat_to_idx[c1], cat_to_idx[c2]] = w
    save_npz(CATEGORY_MTX_FILE, cooc_matrix.tocsr())
    print(f'  💾 Đã lưu category_cooccurrence_matrix: {n_cats}x{n_cats}')

print('✅ Stage 3B hoàn thành!')
all_w = list(normalized_category_weights.values())
print(f'  Tổng cặp: {len(all_w):,} | min={min(all_w):.4f} | max={max(all_w):.4f} | mean={sum(all_w)/len(all_w):.4f}')

=== STAGE 3B: Tính Category Co-occurrence Weights ===
  ⏳ Chưa có: normalized_category_weights.npy
  Đang tính co-occurrence trên outfit_data_filtered...
  Đang tính edge weights...
  💾 Đã lưu normalized_category_weights: 1,022 pairs
  💾 Đã lưu category_cooccurrence_matrix: 59x59
✅ Stage 3B hoàn thành!
  Tổng cặp: 1,022 | min=0.0000 | max=1.0000 | mean=0.0086


## Stage 3C — Item-Item matrix

In [8]:
print('=== STAGE 3C: Xây dựng ma trận Item-Item ===')

ITEM_ITEM_FILE = OUTPUT_PATH + 'matrices/item_item_matrix.npz'

if file_exists(ITEM_ITEM_FILE):
    item_item_matrix = load_npz(ITEM_ITEM_FILE)
    print(f'  item_item_matrix shape: {item_item_matrix.shape}')
else:
    if 'normalized_category_weights' not in dir():
        normalized_category_weights = np.load(CATEGORY_W_FILE, allow_pickle=True).item()

    item_data_str = item_data.copy()
    item_data_str['item_id'] = item_data_str['item_id'].astype(str)
    item_category_map = dict(zip(item_data_str['item_id'], item_data_str['category']))

    category_graph = nx.Graph()
    for (c1, c2), w in normalized_category_weights.items():
        if w > 0:
            category_graph.add_edge(c1, c2, weight=w)

    # Chỉ dùng item trong item_data (đã filtered)
    all_items   = sorted(item_data['item_id'].unique())
    n_items     = len(all_items)
    item_to_idx = {item: idx for idx, item in enumerate(all_items)}

    item_item_matrix = dok_matrix((n_items, n_items), dtype=np.float32)

    total = len(outfit_data)
    for idx_o, row in outfit_data.iterrows():
        items = [str(x) for x in parse_ids(row['items'])]
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                cat1 = item_category_map.get(items[i])
                cat2 = item_category_map.get(items[j])
                if cat1 and cat2 and category_graph.has_edge(cat1, cat2):
                    w = normalized_category_weights.get(
                        (cat1, cat2),
                        normalized_category_weights.get((cat2, cat1), 0.1)
                    )
                    try:
                        i_idx = item_to_idx[int(items[i])]
                        j_idx = item_to_idx[int(items[j])]
                        item_item_matrix[i_idx, j_idx] = w
                        item_item_matrix[j_idx, i_idx] = w
                    except (KeyError, ValueError):
                        pass
        if (idx_o + 1) % 1000 == 0:
            print(f'  [{idx_o+1}/{total}] outfits processed...')

    item_item_matrix = item_item_matrix.tocsr()
    save_npz(ITEM_ITEM_FILE, item_item_matrix)
    print(f'  💾 Đã lưu item_item_matrix: {item_item_matrix.shape}')

print('✅ Stage 3C hoàn thành!')
print(f'\n📋 Item-Item matrix check:')
print(f'  shape  : {item_item_matrix.shape}')
print(f'  nnz    : {item_item_matrix.nnz:,}')
print(f'  density: {item_item_matrix.nnz / (item_item_matrix.shape[0]**2):.6f}')

=== STAGE 3C: Xây dựng ma trận Item-Item ===
  ⏳ Chưa có: item_item_matrix.npz
  [1000/6622] outfits processed...
  [2000/6622] outfits processed...
  [3000/6622] outfits processed...
  [4000/6622] outfits processed...
  [5000/6622] outfits processed...
  [6000/6622] outfits processed...
  💾 Đã lưu item_item_matrix: (14419, 14419)
✅ Stage 3C hoàn thành!

📋 Item-Item matrix check:
  shape  : (14419, 14419)
  nnz    : 73,314
  density: 0.000353


## Tóm tắt toàn bộ Notebook 2

In [9]:
print('=== TỔNG KẾT Notebook 2 ===')
print(f'  Data source     : {SUB_DIR} (filtered từ Notebook 1)')
print(f'  Items           : {len(item_data):,}')
print(f'  Outfits         : {len(outfit_data):,}')
print(f'  Users           : {len(user_data):,}')
print(f'')
print(f'  user_outfit_adj : {user_outfit_adj.shape}  nnz={user_outfit_adj.nnz:,}')
print(f'  outfit_item_adj : {outfit_item_adj.shape} nnz={outfit_item_adj.nnz:,}')
print(f'  item_item_matrix: {item_item_matrix.shape} nnz={item_item_matrix.nnz:,}')
print('\n✅ Notebook 2 hoàn thành! Sẵn sàng cho Notebook 3 (training).')

=== TỔNG KẾT Notebook 2 ===
  Data source     : /kaggle/input/notebooks/kiettruonglifeez/fgat-session1-active-user/subsample/ (filtered từ Notebook 1)
  Items           : 14,419
  Outfits         : 6,622
  Users           : 127,536

  user_outfit_adj : (25263, 6622)  nnz=127,536
  outfit_item_adj : (6622, 14419) nnz=26,047
  item_item_matrix: (14419, 14419) nnz=73,314

✅ Notebook 2 hoàn thành! Sẵn sàng cho Notebook 3 (training).
